### Key Intuition: Why Simple Duplication (Over-Sampling) Fails and Why SMOTE Exists

- **The Problem with `RandomOverSampler`:** It literally copies and pastes existing minority rows. If you duplicate a single point 20 times, the model builds a tiny, rigid boundary around that exact identical point, leading to heavy **overfitting**.
    
- **The Solution — SMOTE (Synthetic Minority Over-sampling Technique):** Instead of duplicating rows, SMOTE creates brand new, synthetic data points **along the line segment between a minority point and its $k$-nearest minority neighbors**.
    
- **The Variation — ADASYN (Adaptive Synthetic):** SMOTE generates equal points everywhere. ADASYN focuses generation specifically on minority samples that are harder to learn (those surrounded by majority class samples near decision boundaries).

# 3. Synthetic Sampling: SMOTE vs. ADASYN

This notebook covers:
1. **Why Synthetic Generation?**: Moving beyond simple duplication to avoid exact-match overfitting.
2. **SMOTE (Synthetic Minority Over-sampling Technique)**: Interpolating new points between $k$-nearest minority neighbors.
3. **ADASYN (Adaptive Synthetic)**: Adaptively focusing synthetic generation on hard-to-learn borderline samples.
4. **Visualizing the Synthetic Geometry**: Plotting 2D feature distributions before and after SMOTE/ADASYN.
5. **Evaluating Performance**: Benchmarking on a realistic, imbalanced test set.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE, ADASYN

# 1. Create a 2D imbalanced classification dataset (90:10 ratio) for easy 2D visualization
X_raw, y_raw = make_classification(
    n_samples=600,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.90, 0.10],
    random_state=42
)

# Convert to DataFrame
df = pd.DataFrame(X_raw, columns=['Feature_1', 'Feature_2'])
df['Target'] = y_raw

print("=== 1. RAW DATASET DISTRIBUTION ===")
display(df['Target'].value_counts().to_frame(name='Count').assign(
    Proportion=df['Target'].value_counts(normalize=True).map('{:.1%}'.format)
))

=== 1. RAW DATASET DISTRIBUTION ===


,Count,Proportion
Target,,
0,538,89.7%
1,62,10.3%


In [2]:
display(df)

,Feature_1,Feature_2,Target
0,-1.777396,0.094918,0
1,-1.405290,0.425859,0
2,-0.659961,1.387649,0
3,-0.670976,1.461806,0
4,-2.002950,-0.421017,0
...,...,...,...
595,-1.604730,1.578392,0
596,-0.608635,0.961252,0
597,-0.327800,1.051398,0
598,-0.912633,0.756672,0


---
## Step 1: Stratified Train / Test Split

As always, split before applying synthetic generation to prevent test leakage.

In [3]:
X = df.drop(columns=['Target']).copy()
y = df['Target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Original Train shape: {X_train.shape} | Majority: {(y_train == 0).sum()} | Minority: {(y_train == 1).sum()}")
print(f"Test shape: {X_test.shape}")

Original Train shape: (450, 2) | Majority: 403 | Minority: 47
Test shape: (150, 2)


---
## Step 2: Applying SMOTE & ADASYN

* **`SMOTE`**: Finds the $k=5$ nearest minority neighbors for each minority point and creates points along the lines between them.
* **`ADASYN`**: Adds a density distribution criterion to generate more points where minority samples are heavily outnumbered by neighboring majority points.